
# Lecture 2 practical: inspecting a real open-weight LLM

This notebook looks inside a small open-weight Qwen model from Hugging Face.

1. Load a Qwen causal language model and its tokenizer.
2. Inspect the configuration and module structure.
3. Connect concrete implementation details to lecture concepts:
   - decoder-only Transformer
   - pre-norm with RMSNorm
   - rotary positional embeddings (RoPE)
   - grouped-query attention (GQA)
   - SwiGLU-style MLP
   - attention / MLP bias choices
   - dropout choices at inference time
   - tied input/output embeddings
4. Tokenize a short prompt and generate a response.
5. Keep and inspect the KV-cache so we can discuss it in class.


In [1]:

# Optional installs (run once if needed)
# %pip install -q "transformers>=4.51.0" accelerate safetensors pandas


In [2]:

import math
import os
from dataclasses import asdict

import pandas as pd
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer


DEVICE = (
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
# bfloat16 works on CPU and MPS and halves memory vs float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.bfloat16

# You can swap this for another open-weight Qwen checkpoint if desired.
# The notebook is written to make that easy.
MODEL_ID = "Qwen/Qwen3-1.7B"

print(f"DEVICE = {DEVICE}")
print(f"DTYPE  = {DTYPE}")
print(f"MODEL  = {MODEL_ID}")


DEVICE = mps
DTYPE  = torch.bfloat16
MODEL  = Qwen/Qwen3-1.7B


## Look at config

It is often useful to inspect the config before loading the full model.

In [3]:
config = AutoConfig.from_pretrained(MODEL_ID)
print(config)
interesting_fields = [
    "model_type",
    "architectures",
    "vocab_size",
    "hidden_size",
    "intermediate_size",
    "num_hidden_layers",
    "num_attention_heads",
    "num_key_value_heads",
    "max_position_embeddings",
    "hidden_act",
    "rms_norm_eps",
    "rope_theta",
    "tie_word_embeddings",
    "attention_bias",
    "mlp_bias",
    "use_cache",
]

rows = []
for key in interesting_fields:
    rows.append({"field": key, "value": getattr(config, key, "<not present>")})

pd.DataFrame(rows)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 6144,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_w

,field,value
0,model_type,qwen3
1,architectures,[Qwen3ForCausalLM]
2,vocab_size,151936
3,hidden_size,2048
4,intermediate_size,6144
5,num_hidden_layers,28
6,num_attention_heads,16
7,num_key_value_heads,8
8,max_position_embeddings,40960
9,hidden_act,silu


### First architectural observations

- `hidden_act` typically reveals the feed-forward nonlinearity, often a **SwiGLU-family** choice in modern LLMs.
- `rms_norm_eps` suggests **RMSNorm** rather than LayerNorm.
- `rope_theta` and related fields indicate **RoPE** rather than absolute learned positional embeddings.
- `tie_word_embeddings` tells us whether the input embedding matrix is reused for the output logits projection.
- `attention_bias` / `mlp_bias` tell us whether the linear maps include bias terms.
- `num_key_value_heads < num_attention_heads` means the model uses **GQA** rather than full multi-head attention with separate K/V per head.

## Load tokenizer and model

This is roughly 4GB download.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)

if DEVICE != "cuda":
    model = model.to(DEVICE)

model.eval()

print(type(model))

`torch_dtype` is deprecated! Use `dtype` instead!
/Users/vary/opt/miniforge3/envs/kv-cache/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/vary/opt/miniforge3/envs/kv-cache/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /Users/vary/opt/miniforge3/envs/kv-cache/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/Users/vary/opt/miniforge3/envs/kv-cache/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/vary/opt/miniforge3/envs/kv-cache/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/vary/opt/miniforge3/envs/kv-cache/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/vary/opt/miniforge3/envs/kv-cache/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functi

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>


## Look at model structure

In [5]:
# show all:
print(model)

# top-level modules only:
print(list(model.named_children()))


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

### Inspect one decoder block

Most of the interesting architecture choices are visible in a single layer.

In [6]:
layers = model.model.layers
layer0 = layers[0]
layer0

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
    (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
    (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
)

### What can we see?

In a modern decoder layer we usually expect:

- input RMSNorm before attention
- self-attention block
- post-attention RMSNorm before MLP
- MLP with gated activation (often SwiGLU)
- residual additions around attention and MLP

This is the **pre-norm** pattern rather than the original post-norm Transformer.

## Count parameters of the model

In [7]:
def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def count_trainable_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

total_params = count_parameters(model)
trainable_params = count_trainable_parameters(model)

print(f"Total parameters:      {total_params:,}")
print(f"Trainable parameters:  {trainable_params:,}")


Total parameters:      1,720,574,976
Trainable parameters:  1,720,574,976


## Extract concrete architecture facts from real modules

In [8]:
attn = layer0.self_attn
mlp = layer0.mlp

facts = {
    "attention module class": type(attn).__name__,
    "mlp module class": type(mlp).__name__,
    "input norm class": type(layer0.input_layernorm).__name__,
    "post-attention norm class": type(layer0.post_attention_layernorm).__name__,
    "q_proj weight shape": tuple(attn.q_proj.weight.shape),
    "k_proj weight shape": tuple(attn.k_proj.weight.shape),
    "v_proj weight shape": tuple(attn.v_proj.weight.shape),
    "o_proj weight shape": tuple(attn.o_proj.weight.shape),
    "gate_proj weight shape": tuple(mlp.gate_proj.weight.shape),
    "up_proj weight shape": tuple(mlp.up_proj.weight.shape),
    "down_proj weight shape": tuple(mlp.down_proj.weight.shape),
}

pd.DataFrame([{"fact": k, "value": str(v)} for k, v in facts.items()])


,fact,value
0,attention module class,Qwen3Attention
1,mlp module class,Qwen3MLP
2,input norm class,Qwen3RMSNorm
3,post-attention norm class,Qwen3RMSNorm
4,q_proj weight shape,"(2048, 2048)"
5,k_proj weight shape,"(1024, 2048)"
6,v_proj weight shape,"(1024, 2048)"
7,o_proj weight shape,"(2048, 2048)"
8,gate_proj weight shape,"(6144, 2048)"
9,up_proj weight shape,"(6144, 2048)"


### Interpreting the projections

These shapes are useful to discuss:

- `q_proj` usually maps from hidden dimension to `num_attention_heads * head_dim`.
- `k_proj` and `v_proj` map to `num_key_value_heads * head_dim`.
- When `num_key_value_heads < num_attention_heads`, keys and values are shared across groups of query heads: **grouped-query attention**.
- `gate_proj`, `up_proj`, `down_proj` are characteristic of **gated MLPs** such as SwiGLU-style feed-forward blocks.


## Prompt formatting, tokenization, and generation

We will use a very small prompt so the tokenization and cache are easy to inspect.

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "In one sentence, explain why RoPE is useful in Transformers."},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(text)


<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
In one sentence, explain why RoPE is useful in Transformers.<|im_end|>
<|im_start|>assistant



In [10]:
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

for k, v in inputs.items():
    print(k, tuple(v.shape), v.dtype, v.device)


input_ids (1, 32) torch.int64 mps:0
attention_mask (1, 32) torch.int64 mps:0


### Decode the prompt tokens so students can see the tokenization

In [ ]:
input_ids = inputs["input_ids"][0]
tokens = [tokenizer.decode([tok]) for tok in input_ids.tolist()]

token_df = pd.DataFrame({
    "position": list(range(len(tokens))),
    "token_id": input_ids.tolist(),
    "decoded_piece": tokens,
})
token_df.head(50)


,position,token_id,decoded_piece
0,0,151644,<|im_start|>
1,1,8948,system
2,2,198,\n
3,3,2610,You
4,4,525,are
5,5,264,a
6,6,10950,helpful
7,7,17847,assistant
8,8,13,.
9,9,151645,<|im_end|>


: 

## Generation

In [ ]:
with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=1.0,
        do_sample=True,
        use_cache=True,
    )

new_tokens = generated[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


## Inspect the KV-cache directly

To make the cache visible, we do a manual forward pass with `use_cache=True`.

For a decoder-only model, after a prefill forward pass over a prompt of length `T`, each layer stores cached keys and values corresponding to those `T` past positions.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, use_cache=True)

type(outputs.past_key_values), len(outputs.past_key_values)


### Examine one layer of the cache

In [ ]:
layer_cache = outputs.past_key_values[0]
print(type(layer_cache))
print(f"Number of tensors stored for one layer: {len(layer_cache)}")
for i, tensor in enumerate(layer_cache):
    print(f"tensor {i}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")


In most Hugging Face decoder models, one layer of `past_key_values` contains:

- cached keys
- cached values

Typical shape convention:

`[batch, num_key_value_heads, seq_len, head_dim]`

This is a very useful concrete object to show in class when explaining autoregressive decoding.

### Summarize KV-cache shapes across all layers

In [ ]:
cache_rows = []
for layer_idx, (k_cache, v_cache) in enumerate(outputs.past_key_values):
    cache_rows.append({
        "layer": layer_idx,
        "k_shape": tuple(k_cache.shape),
        "v_shape": tuple(v_cache.shape),
        "k_dtype": str(k_cache.dtype),
        "v_dtype": str(v_cache.dtype),
    })

cache_df = pd.DataFrame(cache_rows)
cache_df.head()


## 11. Show that the cache grows during decoding

We now do **one manual next-token step** using the existing cache.

Pedagogically this is useful because it makes the purpose of the cache very concrete:

- the prompt is processed once in the prefill stage;
- then each next decoding step only feeds the newly generated token;
- the model reuses the cached K/V tensors from previous positions.


In [ ]:
# Greedy next token from the prefill pass
next_token_id = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
print("Next token id:", next_token_id.item())
print("Decoded token:", repr(tokenizer.decode(next_token_id[0].tolist())))


In [ ]:
with torch.no_grad():
    step_outputs = model(
        input_ids=next_token_id,
        past_key_values=outputs.past_key_values,
        use_cache=True,
    )

old_k = outputs.past_key_values[0][0]
new_k = step_outputs.past_key_values[0][0]

print("Old first-layer K cache shape:", tuple(old_k.shape))
print("New first-layer K cache shape:", tuple(new_k.shape))
print("Sequence length increase:", new_k.shape[-2] - old_k.shape[-2])


The key observation is that the cache sequence dimension increases by exactly 1 after one decode step.

## 12. Small helper: architecture summary for lecture discussion

In [ ]:
summary = {
    "Model ID": MODEL_ID,
    "Decoder-only": True,
    "Number of layers": config.num_hidden_layers,
    "Hidden size": config.hidden_size,
    "Intermediate size": config.intermediate_size,
    "Attention heads": config.num_attention_heads,
    "KV heads": getattr(config, "num_key_value_heads", None),
    "Uses GQA": getattr(config, "num_key_value_heads", config.num_attention_heads) < config.num_attention_heads,
    "Norm type": type(layer0.input_layernorm).__name__,
    "Activation": getattr(config, "hidden_act", None),
    "RoPE theta": getattr(config, "rope_theta", None),
    "Tie embeddings": getattr(config, "tie_word_embeddings", None),
    "Attention bias": getattr(config, "attention_bias", None),
    "MLP bias": getattr(config, "mlp_bias", None),
    "Uses cache": getattr(config, "use_cache", None),
}

pd.DataFrame(summary.items(), columns=["feature", "value"])


## 13. Optional exercises for students

1. Change `MODEL_ID` to another small open-weight causal LM and compare the config.
2. Check whether that model uses RoPE, ALiBi, or learned positional embeddings.
3. Compare `num_attention_heads` and `num_key_value_heads` and decide whether the model uses MHA, MQA, or GQA.
4. Compare tied versus untied embeddings.
5. Change the prompt, inspect the tokenization, and observe how the cache size depends on prompt length.
